In [ ]:
# Unity Catalog
CATALOG_NAME = "nyc_taxi_leonardoaguilera"
RAW_SCHEMA = "raw"
TRUSTED_SCHEMA = "trusted"
REFINED_SCHEMA = "refined"

# Fuentes
YELLOW_TAXI_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
TAXI_ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# Rutas temporales en UC Volume (compatible con shared clusters)
TMP_VOLUME = f"{CATALOG_NAME}.{RAW_SCHEMA}.tmp_data"
TMP_VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{RAW_SCHEMA}/tmp_data"
YELLOW_TAXI_LOCAL = f"{TMP_VOLUME_PATH}/yellow_tripdata_2023-01.parquet"
TAXI_ZONE_LOCAL = f"{TMP_VOLUME_PATH}/taxi_zone_lookup.csv"

# Tablas
RAW_TAXI_TABLE = f"{CATALOG_NAME}.{RAW_SCHEMA}.yellow_taxi_trips"
RAW_ZONES_TABLE = f"{CATALOG_NAME}.{RAW_SCHEMA}.taxi_zone_lookup"
TRUSTED_CLEAN_TABLE = f"{CATALOG_NAME}.{TRUSTED_SCHEMA}.yellow_taxi_trips_clean"
TRUSTED_REJECTED_TABLE = f"{CATALOG_NAME}.{TRUSTED_SCHEMA}.yellow_taxi_trips_rejected"
REFINED_KPI_DEMAND = f"{CATALOG_NAME}.{REFINED_SCHEMA}.kpi_demand_pattern"
REFINED_KPI_EFFICIENCY = f"{CATALOG_NAME}.{REFINED_SCHEMA}.kpi_economic_efficiency"
REFINED_KPI_QUALITY_IMPACT = f"{CATALOG_NAME}.{REFINED_SCHEMA}.kpi_data_quality_impact"
REFINED_DQ_REPORT = f"{CATALOG_NAME}.{REFINED_SCHEMA}.data_quality_report"
REFINED_EXEC_REPORT = f"{CATALOG_NAME}.{REFINED_SCHEMA}.pipeline_execution_report"

# Umbrales de outliers
MAX_TRIP_DISTANCE_MILES = 200
MAX_FARE_AMOUNT = 500
MAX_TRIP_DURATION_MINUTES = 300
MIN_TRIP_DURATION_MINUTES = 1
MAX_PASSENGER_COUNT = 6
MIN_TOTAL_AMOUNT = 0

# Franjas horarias
TIME_SLOTS = {
    "Madrugada (00-04h)": (0, 4),
    "Mañana Temprana (04-08h)": (4, 8),
    "Mañana (08-12h)": (8, 12),
    "Tarde (12-16h)": (12, 16),
    "Tarde-Noche (16-20h)": (16, 20),
    "Noche (20-24h)": (20, 24)
}

# Renombrar a snake_case
COLUMN_RENAME_MAP = {
    "VendorID": "vendor_id",
    "RatecodeID": "ratecode_id",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id"
}

# Periodo esperado
EXPECTED_YEAR = 2023
EXPECTED_MONTH = 1

# Reintentos de descarga
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = [5, 15, 45]

PIPELINE_VERSION = "1.0.0"

# Table properties de gobierno (owner es reservada en UC, se asigna automaticamente)
TABLE_PROPERTIES_BASE = {
    "managed_by": "leonardoaguilera",
    "source_system": "nyc_tlc",
    "pipeline_version": PIPELINE_VERSION
}

TABLE_PROPERTIES = {
    "raw": {**TABLE_PROPERTIES_BASE, "data_classification": "raw", "quality_tier": "raw", "retention_days": "730"},
    "trusted": {**TABLE_PROPERTIES_BASE, "data_classification": "internal", "quality_tier": "trusted", "retention_days": "365"},
    "refined": {**TABLE_PROPERTIES_BASE, "data_classification": "internal", "quality_tier": "refined", "retention_days": "365"}
}

# Schema esperado del parquet de Yellow Taxi
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, TimestampType, StringType

EXPECTED_TAXI_SCHEMA = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])